# **DATA CLEANING**

### **DATASET 1 (Human Development Index)**

In [ ]:
pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 26.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import pycountry
import numpy as np

In [ ]:
df = pd.read_csv("HDR_Table1.csv", encoding="latin1", na_values=['..', 'NA', ''])
df.columns = df.columns.str.strip()

In [ ]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

cols_to_drop = ['1990', 'HDI rank']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])


df.columns

Index(['Country', '2000', '2010', '2015', '2020', '2021', '2022', '2023'], dtype='object')

In [ ]:
df.head(10)

,Country,2000,2010,2015,2020,2021,2022,2023
0,Iceland,0.902,0.935,0.956,0.965,0.967,0.964,0.972
1,Norway,0.924,0.947,0.959,0.969,0.969,0.967,0.970
2,Switzerland,0.892,0.945,0.957,0.963,0.968,0.966,0.970
3,Denmark,0.896,0.920,0.943,0.954,0.958,0.959,0.962
4,Germany,0.897,0.936,0.948,0.955,0.958,0.955,0.959
5,Sweden,0.912,0.918,0.945,0.951,0.958,0.959,0.959
6,Australia,0.897,0.929,0.938,0.950,0.954,0.952,0.958
7,"Hong Kong, China (SAR)",0.839,0.912,0.937,0.950,0.961,0.950,0.955
8,Netherlands,0.900,0.925,0.940,0.945,0.951,0.953,0.955
9,Belgium,0.894,0.921,0.933,0.939,0.948,0.945,0.951


In [ ]:
country_corrections = {
    'Hong Kong, China (SAR)': 'Hong Kong',
    'Korea (Republic of)': 'South Korea',
    'Iran (Islamic Republic of)': 'Iran',
    'Moldova (Republic of)': 'Moldova',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Eswatini (Kingdom of)': 'Eswatini',
    'Micronesia (Federated States of)': 'Micronesia',
    'Tanzania (United Republic of)': 'Tanzania',
    'Congo (Democratic Republic of the)': 'Congo, The Democratic Republic of the'
}

df['Country'] = df['Country'].replace(country_corrections)

In [ ]:
def get_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3.upper()
    except:
        return None

df['ISO'] = df['Country'].apply(get_iso3)

cols = df.columns.tolist()
cols.insert(0, cols.pop(cols.index('ISO')))
df = df[cols]

In [ ]:
df.isnull().sum()

,0
ISO,1
Country,0
2000,22
2010,2
2015,1
2020,1
2021,1
2022,1
2023,0


In [ ]:
year_cols = [col for col in df.columns if col not in ['ISO', 'Country']]

df[year_cols] = df[year_cols].replace('..', np.nan).astype(float)
df[year_cols] = df[year_cols].ffill(axis=1).bfill(axis=1)

print(df[year_cols].isnull().sum())

2000    0
2010    0
2015    0
2020    0
2021    0
2022    0
2023    0
dtype: int64


In [ ]:
df.to_csv("HDI.csv", index=False)

In [ ]:
df.head()

,ISO,Country,2000,2010,2015,2020,2021,2022,2023
0,ISL,Iceland,0.902,0.935,0.956,0.965,0.967,0.964,0.972
1,NOR,Norway,0.924,0.947,0.959,0.969,0.969,0.967,0.970
2,CHE,Switzerland,0.892,0.945,0.957,0.963,0.968,0.966,0.970
3,DNK,Denmark,0.896,0.920,0.943,0.954,0.958,0.959,0.962
4,DEU,Germany,0.897,0.936,0.948,0.955,0.958,0.955,0.959


### **DATASET 2 (CO₂ emissions per capita.)**

In [ ]:
df2 = pd.read_csv("co2_pcap_cons.csv", encoding="latin1", na_values=['..', 'NA', ''])
df2.columns = df2.columns.str.strip()

In [ ]:
df2.head(5)

In [ ]:
df2.rename(columns={'name': 'Country', 'ï»¿geo': 'ISO'}, inplace=True)
df2['ISO'] = df2['ISO'].str.upper()


In [ ]:
year_cols = [col for col in df2.columns if col not in ['ISO', 'Country']]

year_cols_to_keep = [col for col in year_cols if col.isdigit() and 2000 <= int(col) <= 2022]

df2 = df2[['ISO', 'Country'] + year_cols_to_keep]

df2['2023'] = df2['2022']

for col in year_cols_to_keep + ['2023']:
    df2[col] = df2[col].astype(int)

In [ ]:
df2.to_csv("co2_percapita.csv", index=False)

### **DATASET 3 (DISASTERS)**

In [ ]:
df3 = pd.read_csv("disasters(EM-DAT Data).csv", encoding="latin1", na_values=['..', 'NA', ''])
df3.columns = df3.columns.str.strip()

In [ ]:
df3.head(2)

,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,No. Affected,No. Homeless,Total Affected,Reconstruction Costs ('000 US$),"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI
0,1999-9388-DJI,No,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,NaN,NaN,DJI,...,100000.0,NaN,100000.0,NaN,NaN,NaN,NaN,NaN,NaN,56.446576
1,1999-9388-SDN,No,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,NaN,NaN,SDN,...,2000000.0,NaN,2000000.0,NaN,NaN,NaN,NaN,NaN,NaN,54.895152


In [ ]:
missing_counts = df3.isnull().sum()

missing_percentage = (missing_counts / len(df3)) * 100

missing_summary = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage': missing_percentage
})

missing_summary

,Missing Values,Percentage
DisNo.,0,0.000000
Historic,0,0.000000
Classification Key,0,0.000000
Disaster Group,0,0.000000
Disaster Subgroup,0,0.000000
Disaster Type,0,0.000000
Disaster Subtype,0,0.000000
External IDs,12323,74.477215
Event Name,11336,68.512027
ISO,0,0.000000


In [ ]:
df3.columns

In [ ]:
cols_to_drop = [
    'DisNo.', 'Historic', 'Classification Key', 'External IDs', 'Event Name',
    'Origin', 'Associated Types', 'River Basin', 'Location', 'Magnitude',
    'Magnitude Scale', 'Start Month', 'Start Day', 'End Day',
    'Latitude', 'Longitude',
    "Reconstruction Costs ('000 US$)",
    "Reconstruction Costs, Adjusted ('000 US$)",
    "Insured Damage ('000 US$)",
    "Insured Damage, Adjusted ('000 US$)",
    "Total Damage, Adjusted ('000 US$)",
    "AID Contribution ('000 US$)"
]

df3 = df3.drop(columns=cols_to_drop, errors='ignore')


In [ ]:
df3['Response_Indicator'] = df3[['OFDA/BHA Response', 'Appeal', 'Declaration']].apply(
    lambda x: 1 if 'Yes' in x.values else 0,
    axis=1
)

In [ ]:
cols_to_drop = [
    'OFDA/BHA Response', 'Appeal', 'Declaration'
]

df3 = df3.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
df3['Total Deaths'] = df3.groupby(['Country', 'Disaster Type'])['Total Deaths'] \
                        .transform(lambda x: x.fillna(x.median()))

df3['Total Affected'] = df3.groupby('Country')['Total Affected'] \
                         .transform(lambda x: x.fillna(x.median()))

df3= df3.sort_values(['Country', 'Start Year'])
df3['CPI'] = df3.groupby('Country')['CPI'].ffill().bfill()


In [ ]:
df3.isnull().sum()

In [ ]:
df3['Total Affected'] = df3['Total Affected'].fillna(df3['Total Affected'].median())
df3['Total Deaths'] = df3.groupby('Disaster Type')['Total Deaths'] \
                         .transform(lambda x: x.fillna(x.median()))
df3['Total Deaths'] = df3['Total Deaths'] \
                         .fillna(df3['Total Deaths'].median())



In [ ]:
df3.isnull().sum()

In [ ]:
df3.to_csv("disasters.csv", index=False)
